# Sympy + Observer Tests

In this notebook, I want to try out Sympy and maybe get close to calculating some jacobian matrices.

In [69]:
import sympy as sym
import numpy as np
import time,asyncio

sym.init_printing(use_unicode=True)

## State and Input

In [31]:
q_w, q_x, q_y, q_z = sym.symbols("q_w, q_x, q_y, q_z")
v_x, v_y, v_z = sym.symbols("v_x, v_y, v_z")
p_x, p_y, p_z = sym.symbols("p_x, p_y, p_z")

a_x, a_y, a_z = sym.symbols('a_x, a_y, a_z')
g_x, g_y, g_z = sym.symbols('g_x, g_y, g_z')

dt = sym.Symbol('\\Delta t')

gravity = sym.Symbol('g_{gravity}')

In [32]:
q = sym.Matrix([q_w, q_x, q_y, q_z])
v = sym.Matrix([v_x, v_y, v_z])
p = sym.Matrix([p_x, p_y, p_z])

u_a = sym.Matrix([a_x, a_y, a_z])
u_g = sym.Matrix([g_x, g_y, g_z])

x = sym.Matrix.vstack(q, v, p)
u = sym.Matrix.vstack(u_a, u_g)

## Helper functions

For Hamilton product, I turn the quaternion into a matrix to allow for matrix multiplication.
Inversion is trivial.
For quaternion normalization, I also added a quick function to handle that

In [33]:
def left_quat_matrix(q_vec):
    qw, qx, qy, qz = q_vec[0], q_vec[1], q_vec[2], q_vec[3]
    
                            #w
                            #x
                            #y
                            #z
    return sym.Matrix([
        [qw, -qx, -qy, -qz], 
        [qx,  qw, -qz,  qy], 
        [qy,  qz,  qw, -qx], 
        [qz, -qy,  qx,  qw]  
    ])

def quat_inv(q_vec):
    return sym.Matrix([q_vec[0], -q_vec[1], -q_vec[2], -q_vec[3]])
    
def quat_norm(q_vec):
    qw, qx, qy, qz = q_vec[0], q_vec[1], q_vec[2], q_vec[3]
    return q_vec / sym.sqrt(q_vec[0]**2 + q_vec[1]**2 + q_vec[2]**2 + q_vec[3]**2)

## State transition function

### Orientation

In [53]:
gyro_rotation = 0.5 * u_g * dt
gyro_rotation_as_quat = sym.Matrix.vstack(sym.Matrix([1]), gyro_rotation)

f_q_unnormed = left_quat_matrix(q) * gyro_rotation_as_quat

# normalization is only needed in the case of actual hardware implementation.
# Since floating point inaccuracy can influence the results, this would assure that the orientation quaternion stays normalized.
# In the case of the theoretical implementation, I don't need to normalize since I assume infinite precision
# f_q = quat_norm(f_q_unnormed)

f_q = f_q_unnormed
f_q

⎡-0.5⋅\Delta t⋅gₓ⋅qₓ - 0.5⋅\Delta t⋅g_y⋅q_y - 0.5⋅\Delta t⋅g_z⋅q_z + q_w⎤
⎢                                                                       ⎥
⎢0.5⋅\Delta t⋅gₓ⋅q_w - 0.5⋅\Delta t⋅g_y⋅q_z + 0.5⋅\Delta t⋅g_z⋅q_y + qₓ ⎥
⎢                                                                       ⎥
⎢0.5⋅\Delta t⋅gₓ⋅q_z + 0.5⋅\Delta t⋅g_y⋅q_w - 0.5⋅\Delta t⋅g_z⋅qₓ + q_y ⎥
⎢                                                                       ⎥
⎣-0.5⋅\Delta t⋅gₓ⋅q_y + 0.5⋅\Delta t⋅g_y⋅qₓ + 0.5⋅\Delta t⋅g_z⋅q_w + q_z⎦

### Position & Velocity

In [59]:
a_rot1 = left_quat_matrix(q) * sym.Matrix([0, a_x, a_y, a_z])
a_rot2 = left_quat_matrix(rot1) * quat_inv(q)

pure_accel = sym.simplify(a_rot2)[1:4, 0]
f_a = pure_accel - sym.Matrix([0, 0, gravity])

f_a

⎡       q_w⋅(aₓ⋅q_w - a_y⋅q_z + a_z⋅q_y) + qₓ⋅(aₓ⋅qₓ + a_y⋅q_y + a_z⋅q_z) + q_y⋅(-aₓ⋅q_y + a_y⋅qₓ + a_
⎢                                                                                                     
⎢       q_w⋅(aₓ⋅q_z + a_y⋅q_w - a_z⋅qₓ) - qₓ⋅(-aₓ⋅q_y + a_y⋅qₓ + a_z⋅q_w) + q_y⋅(aₓ⋅qₓ + a_y⋅q_y + a_z
⎢                                                                                                     
⎣-g_{gravity} + q_w⋅(-aₓ⋅q_y + a_y⋅qₓ + a_z⋅q_w) + qₓ⋅(aₓ⋅q_z + a_y⋅q_w - a_z⋅qₓ) - q_y⋅(aₓ⋅q_w - a_y⋅

z⋅q_w) - q_z⋅(aₓ⋅q_z + a_y⋅q_w - a_z⋅qₓ)        ⎤
                                                ⎥
⋅q_z) + q_z⋅(aₓ⋅q_w - a_y⋅q_z + a_z⋅q_y)        ⎥
                                                ⎥
q_z + a_z⋅q_y) + q_z⋅(aₓ⋅qₓ + a_y⋅q_y + a_z⋅q_z)⎦

In [55]:
f_v = v + f_a * dt
f_p = p + (v + f_a * dt) * dt

f_v, f_p

⎛⎡        \Delta t⋅(q_w⋅(aₓ⋅q_w - a_y⋅q_z + a_z⋅q_y) + qₓ⋅(aₓ⋅qₓ + a_y⋅q_y + a_z⋅q_z) + q_y⋅(-aₓ⋅q_y +
⎜⎢                                                                                                    
⎜⎢       \Delta t⋅(q_w⋅(aₓ⋅q_z + a_y⋅q_w - a_z⋅qₓ) - qₓ⋅(-aₓ⋅q_y + a_y⋅qₓ + a_z⋅q_w) + q_y⋅(aₓ⋅qₓ + a_
⎜⎢                                                                                                    
⎝⎣\Delta t⋅(-g_{gravity} + q_w⋅(-aₓ⋅q_y + a_y⋅qₓ + a_z⋅q_w) + qₓ⋅(aₓ⋅q_z + a_y⋅q_w - a_z⋅qₓ) - q_y⋅(aₓ

 a_y⋅qₓ + a_z⋅q_w) - q_z⋅(aₓ⋅q_z + a_y⋅q_w - a_z⋅qₓ)) + vₓ        ⎤  ⎡        \Delta t⋅(\Delta t⋅(q_w⋅
                                                                  ⎥  ⎢                                
y⋅q_y + a_z⋅q_z) + q_z⋅(aₓ⋅q_w - a_y⋅q_z + a_z⋅q_y)) + v_y        ⎥, ⎢       \Delta t⋅(\Delta t⋅(q_w⋅(
                                                                  ⎥  ⎢                                
⋅q_w - a_y⋅q_z + a_z⋅q_y) + q_z⋅(aₓ⋅qₓ + a_y⋅q_y + a_z⋅q_z)) + v_z⎦  ⎣\D

### Combined

In [56]:
f_state_transition = sym.Matrix.vstack(f_q, f_v, f_p)
f_state_transition

⎡                                                        -0.5⋅\Delta t⋅gₓ⋅qₓ - 0.5⋅\Delta t⋅g_y⋅q_y - 
⎢                                                                                                     
⎢                                                        0.5⋅\Delta t⋅gₓ⋅q_w - 0.5⋅\Delta t⋅g_y⋅q_z + 
⎢                                                                                                     
⎢                                                        0.5⋅\Delta t⋅gₓ⋅q_z + 0.5⋅\Delta t⋅g_y⋅q_w - 
⎢                                                                                                     
⎢                                                        -0.5⋅\Delta t⋅gₓ⋅q_y + 0.5⋅\Delta t⋅g_y⋅qₓ + 
⎢                                                                                                     
⎢                \Delta t⋅(q_w⋅(aₓ⋅q_w - a_y⋅q_z + a_z⋅q_y) + qₓ⋅(aₓ⋅qₓ + a_y⋅q_y + a_z⋅q_z) + q_y⋅(-a
⎢                                                                        

## Measurement function

In [65]:
h_rot1 = left_quat_matrix(quat_inv(q)) * sym.Matrix([0, 0, 0, gravity])
h_rot2 = left_quat_matrix(h_rot1) * q

h_measurement = h_rot2[1:4, 0]
h_measurement

⎡             -2⋅g_{gravity}⋅q_w⋅q_y + 2⋅g_{gravity}⋅qₓ⋅q_z              ⎤
⎢                                                                        ⎥
⎢              2⋅g_{gravity}⋅q_w⋅qₓ + 2⋅g_{gravity}⋅q_y⋅q_z              ⎥
⎢                                                                        ⎥
⎢               2                 2                  2                  2⎥
⎣g_{gravity}⋅q_w  - g_{gravity}⋅qₓ  - g_{gravity}⋅q_y  + g_{gravity}⋅q_z ⎦

## Jacobian Matrices

In [57]:
A = f_state_transition.jacobian(x)
A

⎡                     1                                     -0.5⋅\Delta t⋅gₓ                          
⎢                                                                                                     
⎢              0.5⋅\Delta t⋅gₓ                                      1                                 
⎢                                                                                                     
⎢              0.5⋅\Delta t⋅g_y                             -0.5⋅\Delta t⋅g_z                         
⎢                                                                                                     
⎢              0.5⋅\Delta t⋅g_z                             0.5⋅\Delta t⋅g_y                          
⎢                                                                                                     
⎢\Delta t⋅(2⋅aₓ⋅q_w - 2⋅a_y⋅q_z + 2⋅a_z⋅q_y)   \Delta t⋅(2⋅aₓ⋅qₓ + 2⋅a_y⋅q_y + 2⋅a_z⋅q_z)    \Delta t⋅
⎢                                                                        

In [67]:
H = h_measurement.jacobian(x)
H

⎡-2⋅g_{gravity}⋅q_y  2⋅g_{gravity}⋅q_z  -2⋅g_{gravity}⋅q_w  2⋅g_{gravity}⋅qₓ   0  0  0  0  0  0⎤
⎢                                                                                              ⎥
⎢ 2⋅g_{gravity}⋅qₓ   2⋅g_{gravity}⋅q_w  2⋅g_{gravity}⋅q_z   2⋅g_{gravity}⋅q_y  0  0  0  0  0  0⎥
⎢                                                                                              ⎥
⎣2⋅g_{gravity}⋅q_w   -2⋅g_{gravity}⋅qₓ  -2⋅g_{gravity}⋅q_y  2⋅g_{gravity}⋅q_z  0  0  0  0  0  0⎦

In [58]:
parameters = [
    g_x, g_y, g_z,
    a_x, a_y, a_z,
    dt, gravity
]

input_symbols = list(x) + parameters

calculate_A_matrix = sym.lambdify(input_symbols, A, 'numpy')
calculate_A_matrix

<function _lambdifygenerated(q_w, q_x, q_y, q_z, v_x, v_y, v_z, p_x, p_y, p_z, g_x, g_y, g_z, a_x, a_y, a_z, _Dummy_38, g_gravity)>

In [71]:
async def count():
    print("count one")
    await asyncio.sleep(1)
    print("count four")

async def count_further():
    print("count two")
    await asyncio.sleep(1)
    print("count five")

async def count_even_further():
    print("count three")
    await asyncio.sleep(1)
    print("count six")

async def main():
    await asyncio.gather(count(), count_further(), count_even_further())

s = time.perf_counter()
await main()
elapsed = time.perf_counter() - s
print(f"Script executed in {elapsed:0.2f} seconds.")



count one
count two
count three
count four
count five
count six
Script executed in 1.00 seconds.
